# 🧠 Демонстрация Dunder (магических) методов в Python
# Тема: __add__ и его работа внутри классов и интерпретатора

---

## 🧩 Что такое Dunder методы

 Dunder методы (от слов double underscore — двойное подчеркивание) — это специальные методы,
 которые позволяют нашим классам вести себя как встроенные типы Python.
 Например, чтобы объект мог участвовать в операциях сложения, вычитания, сравнения, вызова и т.д.
 Все они начинаются и заканчиваются двумя подчёркиваниями: __init__, __add__, __len__, __str__, __eq__ и т.д.
 Они не вызываются напрямую пользователем (хотя можно и так), а Python вызывает их сам
 при определённых операциях.
 Пример: когда мы пишем a + b, на самом деле Python вызывает a.__add__(b)


---

## ⚙️ Как работает __add__

 Метод __add__(self, other) используется для определения поведения оператора +
 между объектами пользовательского класса.
 Например, если мы хотим сложить два вектора, две матрицы, строки, или даже свои кастомные объекты.


### 📘 Пример 1: Базовая реализация

In [2]:

class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # Реализуем dunder метод сложения
    def __add__(self, other):
        # Проверим, что other — это тоже Vector
        if not isinstance(other, Vector):
            return NotImplemented  # Python попробует вызвать __radd__ другого объекта
        return Vector(self.x + other.x, self.y + other.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

# Проверим работу
v1 = Vector(2, 5)
v2 = Vector(3, 4)
print(v1 + v2)  # Vector(5, 9)

Vector(5, 9)


# Python под капотом делает: v1.__add__(v2)


---

### 📘 Пример 2: Поведение при разных типах

# Когда мы пытаемся сложить объект с чем-то несовместимым

In [3]:
print(v1 + 10) # Выдаст ошибку, потому что возвращён NotImplemented

TypeError: unsupported operand type(s) for +: 'Vector' and 'int'



# Что делает Python в этот момент:
 1️⃣ вызывает v1.__add__(10) → получает NotImplemented

 2️⃣ если у второго объекта (10) есть __radd__, пробует вызвать 10.__radd__(v1)

 3️⃣ если и это не сработало — выбрасывает TypeError


---

# 🧩 Разница между __add__ и __radd__

# Когда Python видит выражение a + b, он делает следующее:
 1️⃣ Сначала вызывает a.__add__(b)

 2️⃣ Если тот вернул NotImplemented, вызывает b.__radd__(a)

 3️⃣ Если и это не сработало, выбрасывает TypeError


In [5]:
class A:
    def __add__(self, other):
        print("A.__add__ called")
        return NotImplemented  # отказываемся обрабатывать сложение

class B:
    def __radd__(self, other):
        print("B.__radd__ called")
        return "handled by B"

a = A()
b = B()

print(a + b)  # Сначала A.__add__, потом B.__radd__

A.__add__ called
B.__radd__ called
handled by B


# Пример с пользовательским классом

In [8]:
class Length:
    def __init__(self, meters):
        self.meters = meters

    def __add__(self, other):
        if isinstance(other, Length):
            return Length(self.meters + other.meters)
        elif isinstance(other, (int, float)):
            return Length(self.meters + other)
        return NotImplemented

    def __radd__(self, other):
        # если левый операнд не знает, как работать с нами
        return self.__add__(other)

    def __repr__(self):
        return f"Length({self.meters}m)"

l1 = Length(5)
print(l1 + 10)  # Length(15m) → вызвался __add__
print(10 + l1)  # Length(15m) → вызвался __radd__


Length(15m)
Length(15m)



# ⚖️ Разница между __add__ и __iadd__

 __add__ используется для операции a + b

__iadd__ используется для операции a += b

 Если __iadd__ не определён, Python просто вызовет __add__ и присвоит результат:


 a = a + b

In [9]:

class Counter:
    def __init__(self, value=0):
        self.value = value

    def __add__(self, other):
        return Counter(self.value + other)

    def __iadd__(self, other):
        # изменяет объект на месте
        self.value += other
        return self

    def __repr__(self):
        return f"Counter({self.value})"

c = Counter(10)
c += 5  # вызовет __iadd__
print(c)  # Counter(15)

c = Counter(10)
c = c + 5  # вызовет __add__, создаст новый объект
print(c)  # Counter(15)


Counter(15)
Counter(15)


---

## 🔍 Что происходит внутри (C уровень)

 Внутри интерпретатора Python (написанного на C), операции вроде a + b
 реализованы через специальные C-функции в файле Objects/abstract.c или listobject.c.

 # Упрощённый псевдокод логики операции +:


```
if (a имеет метод nb_add):
    result = a.nb_add(a, b)
    if result != NotImplemented:
        return result

if (b имеет метод nb_add):
    result = b.nb_add(b, a)
    if result != NotImplemented:
        return result

raise TypeError("unsupported operand type(s) for +")
```


 Для списка list этот nb_add указывает на функцию list_concat() в C,
 которая создаёт новый список и копирует в него элементы из обоих списков.



---
## 🧠 Итог

 🔹 Dunder методы позволяют нашим объектам вести себя как встроенные типы Python.
 🔹 __add__ — ключевой метод для перегрузки оператора +.
 🔹 Если вернуть NotImplemented, Python попробует вызвать __radd__.
 🔹 Для += лучше реализовывать __iadd__, чтобы не создавать новый объект.
 🔹 Все эти методы «магически» вызываются самим Python, но на уровне C они реализованы
 через таблицу указателей (PyNumberMethods) для каждого типа.

# Таким образом, магия Python — это просто хорошо спрятанная структура из C-кода.
